# NodPT Prompt & Sample Tests

This notebook covers the three test modules in `AI/src/`:

1. **`tests/test_formats.py`** — Unit tests for the Ollama JSON Schema files (`format.json`) across all four node types. Also validates prompt file structure and Ollama payload construction.
2. **`tests/test_generate_samples.py`** — Unit tests for the sample validation and parsing logic in `generate_samples.py` (26 tests covering edge cases).
3. **`test_vllm_samples.py`** — Functional tests for `generate_vllm_samples.py` covering random prompt generation, batch prompt building, JSONL parsing, and Markdown fence stripping.

---

## Test Infrastructure

| Test file | Framework | Run command |
|-----------|-----------|-------------|
| `tests/test_formats.py` | `unittest` | `python -m unittest tests.test_formats -v` |
| `tests/test_generate_samples.py` | `unittest` | `python -m unittest tests.test_generate_samples -v` |
| `test_vllm_samples.py` | Plain assert | `python test_vllm_samples.py` |

All commands are run from `AI/src/`.

---

## Prerequisites

```bash
cd AI/src
pip install -r requirements.txt
```

No running Ollama or vLLM server is needed — all tests run offline.

---
## Part 1 — Node Format Tests (`tests/test_formats.py`)

**Script:** `AI/src/tests/test_formats.py`

These tests guard the integrity of the JSON Schema files that define each node type's structured output format. The schemas are used both at inference time (passed to Ollama's `format` field) and as validation references during training data generation.

### What is tested
- Each schema is a valid JSON object type with the correct `type`, `properties`, and `required` fields.
- Node-type-specific array fields are present and correctly typed.
- Array item schemas contain the required sub-fields.
- The node hierarchy is clean — no schema leaks fields from other node types.
- Prompt files exist and are non-empty.
- Full Ollama API payload structure is valid and serialisable.

### 1.1 Test Setup — Format Loader and Node Types

In [ ]:
import json
import os
import unittest

# ── Resolve base directory for the test suite ─────────────────────────────────
# In the actual test file this resolves to AI/src/ via __file__.
# When running in this notebook, adjust accordingly.
BASE_DIR   = os.path.abspath("AI/src")
NODE_TYPES = ["Director", "Manager", "Supervisor", "Agent"]


def load_format(node_type: str) -> dict:
    """
    Load the format.json schema for a node type from AI/src/<NodeType>/.

    Args:
        node_type: Capitalised node type name (e.g. 'Director').

    Returns:
        Parsed JSON Schema dict.

    Raises:
        FileNotFoundError: if the format.json file does not exist.
    """
    path = os.path.join(BASE_DIR, node_type, "format.json")
    with open(path, "r") as f:
        return json.load(f)


# Verify all format files are reachable before running tests
for nt in NODE_TYPES:
    fmt = load_format(nt)
    print(f"{nt}: type={fmt['type']}, required={fmt['required']}")

### 1.2 TestDirectorFormat — Director Schema Validation

The Director schema must define `content` (string) and `managers` (array of objects with `name`/`job` fields). Both must be in the `required` list.

In [ ]:
class TestDirectorFormat(unittest.TestCase):
    """Validate the Director node type JSON Schema."""

    def setUp(self):
        """Load the Director schema once before each test method."""
        self.fmt = load_format("Director")

    def test_is_object_type(self):
        """Schema root must be an object type (Ollama requirement)."""
        self.assertEqual(self.fmt["type"], "object")

    def test_has_content_property(self):
        """'content' must be a string property — used for the conversational response."""
        self.assertIn("content", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["content"]["type"], "string")

    def test_has_managers_array(self):
        """Director outputs a 'managers' array — its top-level delegation field."""
        self.assertIn("managers", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["managers"]["type"], "array")

    def test_manager_items_have_name_and_job(self):
        """Each manager item must have string 'name' and 'job' properties."""
        items = self.fmt["properties"]["managers"]["items"]
        self.assertEqual(items["type"], "object")
        self.assertIn("name", items["properties"])
        self.assertIn("job",  items["properties"])
        self.assertEqual(items["properties"]["name"]["type"], "string")
        self.assertEqual(items["properties"]["job"]["type"],  "string")

    def test_required_fields(self):
        """Both 'content' and 'managers' must appear in the 'required' list."""
        self.assertIn("content",  self.fmt["required"])
        self.assertIn("managers", self.fmt["required"])

    def test_manager_items_required_fields(self):
        """Each manager item must require both 'name' and 'job'."""
        items = self.fmt["properties"]["managers"]["items"]
        self.assertIn("name", items["required"])
        self.assertIn("job",  items["required"])


# Run the suite and display results inline
suite = unittest.TestLoader().loadTestsFromTestCase(TestDirectorFormat)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 1.3 TestManagerFormat — Manager Schema Validation

The Manager schema uses `supervisors` as its delegation array instead of `managers`. The item fields remain `name` and `job`.

In [ ]:
class TestManagerFormat(unittest.TestCase):
    """Validate the Manager node type JSON Schema."""

    def setUp(self):
        self.fmt = load_format("Manager")

    def test_is_object_type(self):
        """Schema root must be an object type."""
        self.assertEqual(self.fmt["type"], "object")

    def test_has_content_property(self):
        """'content' must be a string property."""
        self.assertIn("content", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["content"]["type"], "string")

    def test_has_supervisors_array(self):
        """Manager outputs a 'supervisors' array — its delegation field."""
        self.assertIn("supervisors", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["supervisors"]["type"], "array")

    def test_supervisor_items_have_name_and_job(self):
        """Each supervisor item must have string 'name' and 'job' properties."""
        items = self.fmt["properties"]["supervisors"]["items"]
        self.assertEqual(items["type"], "object")
        self.assertIn("name", items["properties"])
        self.assertIn("job",  items["properties"])
        self.assertEqual(items["properties"]["name"]["type"], "string")
        self.assertEqual(items["properties"]["job"]["type"],  "string")

    def test_required_fields(self):
        """Both 'content' and 'supervisors' must be required."""
        self.assertIn("content",     self.fmt["required"])
        self.assertIn("supervisors", self.fmt["required"])

    def test_supervisor_items_required_fields(self):
        """Each supervisor item must require 'name' and 'job'."""
        items = self.fmt["properties"]["supervisors"]["items"]
        self.assertIn("name", items["required"])
        self.assertIn("job",  items["required"])


suite = unittest.TestLoader().loadTestsFromTestCase(TestManagerFormat)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 1.4 TestSupervisorFormat and TestAgentFormat

- **Supervisor** uses `agents` as its delegation array (items: `name`, `job`).
- **Agent** uses `files` as its output array (items: `filename`, `content`) — note the different item field names reflecting that agents produce actual source code files.

In [ ]:
class TestSupervisorFormat(unittest.TestCase):
    """Validate the Supervisor node type JSON Schema."""

    def setUp(self):
        self.fmt = load_format("Supervisor")

    def test_is_object_type(self):
        """Schema root must be an object type."""
        self.assertEqual(self.fmt["type"], "object")

    def test_has_content_property(self):
        """'content' must be a string property."""
        self.assertIn("content", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["content"]["type"], "string")

    def test_has_agents_array(self):
        """Supervisor outputs an 'agents' array — its delegation field."""
        self.assertIn("agents", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["agents"]["type"], "array")

    def test_agent_items_have_name_and_job(self):
        """Each agent item must have string 'name' and 'job' properties."""
        items = self.fmt["properties"]["agents"]["items"]
        self.assertEqual(items["type"], "object")
        self.assertIn("name", items["properties"])
        self.assertIn("job",  items["properties"])
        self.assertEqual(items["properties"]["name"]["type"], "string")
        self.assertEqual(items["properties"]["job"]["type"],  "string")

    def test_required_fields(self):
        """Both 'content' and 'agents' must be required."""
        self.assertIn("content", self.fmt["required"])
        self.assertIn("agents",  self.fmt["required"])

    def test_agent_items_required_fields(self):
        """Each agent item must require 'name' and 'job'."""
        items = self.fmt["properties"]["agents"]["items"]
        self.assertIn("name", items["required"])
        self.assertIn("job",  items["required"])


class TestAgentFormat(unittest.TestCase):
    """Validate the Agent node type JSON Schema."""

    def setUp(self):
        self.fmt = load_format("Agent")

    def test_is_object_type(self):
        """Schema root must be an object type."""
        self.assertEqual(self.fmt["type"], "object")

    def test_has_content_property(self):
        """'content' must be a string property."""
        self.assertIn("content", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["content"]["type"], "string")

    def test_has_files_array(self):
        """Agent outputs a 'files' array — the actual code files it produces."""
        self.assertIn("files", self.fmt["properties"])
        self.assertEqual(self.fmt["properties"]["files"]["type"], "array")

    def test_file_items_have_filename_and_content(self):
        """Each file item must have 'filename' and 'content' string properties."""
        items = self.fmt["properties"]["files"]["items"]
        self.assertEqual(items["type"], "object")
        self.assertIn("filename", items["properties"])
        self.assertIn("content",  items["properties"])
        self.assertEqual(items["properties"]["filename"]["type"], "string")
        self.assertEqual(items["properties"]["content"]["type"],  "string")

    def test_required_fields(self):
        """Both 'content' and 'files' must be required."""
        self.assertIn("content", self.fmt["required"])
        self.assertIn("files",   self.fmt["required"])

    def test_file_items_required_fields(self):
        """Each file item must require 'filename' and 'content'."""
        items = self.fmt["properties"]["files"]["items"]
        self.assertIn("filename", items["required"])
        self.assertIn("content",  items["required"])


# Run both suites
for cls in [TestSupervisorFormat, TestAgentFormat]:
    suite = unittest.TestLoader().loadTestsFromTestCase(cls)
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

### 1.5 TestNodeHierarchy — Cross-Type Isolation Tests

These tests enforce that each node type schema contains **only** its own delegation array and does not accidentally include fields from other levels of the hierarchy. This protects against schema drift across the four node types.

In [ ]:
class TestNodeHierarchy(unittest.TestCase):
    """Cross-node-type tests verifying schema isolation and shared 'content' field."""

    def test_all_formats_are_objects(self):
        """Every node type schema must declare type='object' (Ollama requirement)."""
        for node_type in NODE_TYPES:
            fmt = load_format(node_type)
            self.assertEqual(fmt["type"], "object", f"{node_type} should be object type")

    def test_all_formats_have_content(self):
        """Every node type must have 'content' as a required string property."""
        for node_type in NODE_TYPES:
            fmt = load_format(node_type)
            self.assertIn("content", fmt["properties"], f"{node_type} missing content")
            self.assertIn("content", fmt["required"],   f"{node_type} content not required")

    def test_director_only_has_managers(self):
        """
        Director schema must have 'managers' but must NOT have 'supervisors',
        'agents', or 'files' — those belong to downstream node types.
        """
        fmt = load_format("Director")
        self.assertIn("managers",    fmt["properties"])
        self.assertNotIn("supervisors", fmt["properties"])
        self.assertNotIn("agents",      fmt["properties"])
        self.assertNotIn("files",       fmt["properties"])

    def test_manager_only_has_supervisors(self):
        """
        Manager schema must have 'supervisors' but must NOT have 'managers',
        'agents', or 'files'.
        """
        fmt = load_format("Manager")
        self.assertNotIn("managers",    fmt["properties"])
        self.assertIn("supervisors",    fmt["properties"])
        self.assertNotIn("agents",      fmt["properties"])
        self.assertNotIn("files",       fmt["properties"])

    def test_supervisor_only_has_agents(self):
        """
        Supervisor schema must have 'agents' but must NOT have 'managers',
        'supervisors', or 'files'.
        """
        fmt = load_format("Supervisor")
        self.assertNotIn("managers",    fmt["properties"])
        self.assertNotIn("supervisors", fmt["properties"])
        self.assertIn("agents",         fmt["properties"])
        self.assertNotIn("files",       fmt["properties"])

    def test_agent_only_has_files(self):
        """
        Agent schema must have 'files' but must NOT have 'managers',
        'supervisors', or 'agents' — Agent is a leaf node that produces files.
        """
        fmt = load_format("Agent")
        self.assertNotIn("managers",    fmt["properties"])
        self.assertNotIn("supervisors", fmt["properties"])
        self.assertNotIn("agents",      fmt["properties"])
        self.assertIn("files",          fmt["properties"])


suite = unittest.TestLoader().loadTestsFromTestCase(TestNodeHierarchy)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 1.6 TestPrompts and TestRequestPayload

`TestPrompts` verifies that each node type has a `prompts/` directory with at least one non-empty `sample.txt` file. `TestRequestPayload` builds a full Ollama API payload and verifies it can be serialised to JSON — this mimics exactly what `run.py` sends to Ollama.

In [ ]:
class TestPrompts(unittest.TestCase):
    """Verify that prompt files exist and are non-empty for all node types."""

    def test_each_node_type_has_prompts_folder(self):
        """
        Each node type directory must contain a 'prompts/' subdirectory.
        This folder is required by run.py and the test_formats test suite.
        """
        for node_type in NODE_TYPES:
            prompts_dir = os.path.join(BASE_DIR, node_type, "prompts")
            self.assertTrue(
                os.path.isdir(prompts_dir),
                f"{node_type}/prompts/ directory should exist",
            )

    def test_each_node_type_has_sample_prompt(self):
        """
        Each node type must have a 'prompts/sample.txt' file.
        This file is used as the default demo prompt in CLI examples.
        """
        for node_type in NODE_TYPES:
            path = os.path.join(BASE_DIR, node_type, "prompts", "sample.txt")
            self.assertTrue(
                os.path.isfile(path),
                f"{node_type}/prompts/sample.txt should exist",
            )

    def test_sample_prompts_are_non_empty(self):
        """
        Each sample.txt must contain at least one non-whitespace character.
        An empty prompt file would silently produce meaningless model output.
        """
        for node_type in NODE_TYPES:
            path = os.path.join(BASE_DIR, node_type, "prompts", "sample.txt")
            with open(path, "r") as f:
                content = f.read().strip()
            self.assertGreater(
                len(content), 0,
                f"{node_type}/prompts/sample.txt should not be empty",
            )


class TestRequestPayload(unittest.TestCase):
    """Verify that Ollama API payloads are correctly structured and serialisable."""

    def test_payload_structure(self):
        """
        Build a full Ollama /api/generate payload for each node type and verify:
          1. All required top-level fields are present (model, prompt, stream, format).
          2. The 'format' field is a valid JSON Schema object.
          3. The entire payload serialises to valid JSON without errors.

        This test simulates exactly what run.py does before calling requests.post().
        """
        for node_type in NODE_TYPES:
            fmt = load_format(node_type)
            prompt_path = os.path.join(BASE_DIR, node_type, "prompts", "sample.txt")
            with open(prompt_path, "r") as f:
                prompt = f.read().strip()

            payload = {
                "model":  "llama3.1:8b",
                "prompt": prompt,
                "stream": False,
                "format": fmt,
            }

            # Required top-level fields
            self.assertIn("model",  payload)
            self.assertIn("prompt", payload)
            self.assertIn("stream", payload)
            self.assertIn("format", payload)

            # Format must be a valid JSON Schema object
            self.assertEqual(payload["format"]["type"], "object")
            self.assertIn("properties", payload["format"])
            self.assertIn("required",   payload["format"])

            # Entire payload must be JSON-serialisable
            serialised = json.dumps(payload)
            self.assertIsInstance(json.loads(serialised), dict)


# Run both suites
for cls in [TestPrompts, TestRequestPayload]:
    suite = unittest.TestLoader().loadTestsFromTestCase(cls)
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

---
## Part 2 — Sample Generation Tests (`tests/test_generate_samples.py`)

**Script:** `AI/src/tests/test_generate_samples.py`

26 unit tests covering the two pure functions from `generate_samples.py` that are testable without a network connection:
- `validate_sample_output` — schema validation of a generated output JSON string.
- `parse_generated_lines` — full pipeline from raw model text to validated sample list.

`aiohttp` is mocked at import time so the tests work without it installed.

### 2.1 Setup — Import Pure Functions with aiohttp Mock

Since `generate_samples.py` imports `aiohttp` at module level (for type hints used by async functions), we stub it out before importing so the test environment doesn't need the package installed.

In [ ]:
import json
import os
import sys
import types
import unittest

# ── Add generate_samples.py to the import path ────────────────────────────────
SCRIPT_DIR = os.path.abspath(os.path.join("AI", "src", "fine-tuning", "scripts"))

# ── Mock aiohttp so the import succeeds without the package installed ──────────
# Only the pure validation/parsing functions are tested; no async/network code
# is executed during tests.
if "aiohttp" not in sys.modules:
    sys.modules["aiohttp"] = types.ModuleType("aiohttp")

if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

from generate_samples import (  # noqa: E402
    NODE_CONFIG,
    parse_generated_lines,
    validate_sample_output,
)

print("Imported validate_sample_output and parse_generated_lines successfully.")
print(f"NODE_CONFIG keys: {list(NODE_CONFIG.keys())}")

### 2.2 TestValidateSampleOutput — Valid Inputs

Tests the 'happy path': valid JSON strings that should pass validation for each node type.

In [ ]:
class TestValidateSampleOutputValid(unittest.TestCase):
    """Tests for validate_sample_output() — valid input cases."""

    def test_valid_director_output(self):
        """A correctly structured Director output must pass validation."""
        output = json.dumps({
            "content": "I'll organise the project into two domains.",
            "managers": [
                {"name": "Backend Manager", "job": "Handle API development."},
                {"name": "Frontend Manager", "job": "Build the UI."},
            ],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertTrue(ok, msg)

    def test_valid_manager_output(self):
        """A correctly structured Manager output must pass validation."""
        output = json.dumps({
            "content": "Breaking down the backend work.",
            "supervisors": [
                {"name": "DB Supervisor",  "job": "Design the database."},
                {"name": "API Supervisor", "job": "Create REST endpoints."},
            ],
        })
        ok, msg = validate_sample_output(output, "manager")
        self.assertTrue(ok, msg)

    def test_valid_supervisor_output(self):
        """A correctly structured Supervisor output must pass validation."""
        output = json.dumps({
            "content": "Assigning coding tasks.",
            "agents": [
                {"name": "Auth Agent", "job": "Write login endpoint."},
                {"name": "DB Agent",   "job": "Write migration script."},
            ],
        })
        ok, msg = validate_sample_output(output, "supervisor")
        self.assertTrue(ok, msg)

    def test_valid_agent_output(self):
        """A correctly structured Agent output must pass validation."""
        output = json.dumps({
            "content": "Created the login controller.",
            "files": [
                {"filename": "login.py",      "content": "print('hello')"},
                {"filename": "test_login.py", "content": "assert True"},
            ],
        })
        ok, msg = validate_sample_output(output, "agent")
        self.assertTrue(ok, msg)

    def test_array_min_boundary(self):
        """Exactly min_items (2) items must be accepted."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [
                {"name": "M1", "job": "J1"},
                {"name": "M2", "job": "J2"},
            ],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertTrue(ok, msg)

    def test_array_max_boundary(self):
        """Exactly max_items (5) items must be accepted."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [{"name": f"M{i}", "job": f"J{i}"} for i in range(5)],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertTrue(ok, msg)


suite = unittest.TestLoader().loadTestsFromTestCase(TestValidateSampleOutputValid)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 2.3 TestValidateSampleOutput — Invalid Inputs

Tests the error paths: malformed JSON, missing fields, wrong types, out-of-range array sizes, and empty strings. Each test checks both the boolean result and the error message content to ensure the rejection reason is descriptive.

In [ ]:
class TestValidateSampleOutputInvalid(unittest.TestCase):
    """Tests for validate_sample_output() — invalid input cases."""

    def test_invalid_json_string(self):
        """Non-JSON input must be rejected with a clear error message."""
        ok, msg = validate_sample_output("not json at all", "director")
        self.assertFalse(ok)
        self.assertIn("not valid JSON", msg)

    def test_json_array_not_object(self):
        """A JSON array root must be rejected — the output must be a JSON object."""
        ok, msg = validate_sample_output("[1, 2, 3]", "director")
        self.assertFalse(ok)
        self.assertIn("not a JSON object", msg)

    def test_missing_content_field(self):
        """Output without 'content' field must be rejected."""
        output = json.dumps({
            "managers": [{"name": "M1", "job": "J1"}, {"name": "M2", "job": "J2"}]
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("content", msg)

    def test_empty_content_field(self):
        """A whitespace-only 'content' field must be rejected."""
        output = json.dumps({
            "content": "   ",
            "managers": [{"name": "M1", "job": "J1"}, {"name": "M2", "job": "J2"}],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("content", msg)

    def test_missing_array_field(self):
        """Output with 'content' but no 'managers' array must be rejected."""
        output = json.dumps({"content": "Plan here."})
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("managers", msg)

    def test_array_too_few_items(self):
        """An array with fewer than min_items (2) items must be rejected."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [{"name": "M1", "job": "J1"}],  # Only 1 item
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("at least", msg)

    def test_array_too_many_items(self):
        """An array with more than max_items (5) items must be rejected."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [{"name": f"M{i}", "job": f"J{i}"} for i in range(6)],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("at most", msg)

    def test_item_missing_required_field(self):
        """An array item that is missing a required sub-field must be rejected."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [
                {"name": "M1"},              # Missing 'job'
                {"name": "M2", "job": "J2"},
            ],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("job", msg)

    def test_item_empty_field(self):
        """An array item with an empty string sub-field must be rejected."""
        output = json.dumps({
            "content": "Plan.",
            "managers": [
                {"name": "", "job": "J1"},   # Empty 'name'
                {"name": "M2", "job": "J2"},
            ],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("empty", msg)

    def test_item_not_object(self):
        """An array item that is a string instead of an object must be rejected."""
        output = json.dumps({
            "content": "Plan.",
            "managers": ["not an object", {"name": "M2", "job": "J2"}],
        })
        ok, msg = validate_sample_output(output, "director")
        self.assertFalse(ok)
        self.assertIn("not an object", msg)

    def test_agent_item_missing_filename(self):
        """An Agent file item missing 'filename' must be rejected."""
        output = json.dumps({
            "content": "Done.",
            "files": [
                {"content": "code"},              # Missing 'filename'
                {"filename": "b.py", "content": "code"},
            ],
        })
        ok, msg = validate_sample_output(output, "agent")
        self.assertFalse(ok)
        self.assertIn("filename", msg)


suite = unittest.TestLoader().loadTestsFromTestCase(TestValidateSampleOutputInvalid)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### 2.4 TestParseGeneratedLines — JSONL Parsing Pipeline

Tests the full parsing pipeline: raw multi-line text → list of validated sample dicts. Includes edge cases like Markdown fences, empty lines, object-vs-string output coercion, and instruction normalisation.

In [ ]:
class TestParseGeneratedLines(unittest.TestCase):
    """Tests for parse_generated_lines()."""

    def _make_line(self, instruction="test", inp="test input", output_obj=None) -> str:
        """
        Helper: build a single valid JSONL line for 'director'.

        Args:
            instruction: Instruction string (will be normalised to canonical value).
            inp:         Input string.
            output_obj:  Output dict (default: minimal valid director output).

        Returns:
            JSON-serialised JSONL line string.
        """
        if output_obj is None:
            output_obj = {
                "content": "Plan.",
                "managers": [
                    {"name": "M1", "job": "J1"},
                    {"name": "M2", "job": "J2"},
                ],
            }
        return json.dumps({
            "instruction": instruction,
            "input":       inp,
            "output":      json.dumps(output_obj),
        })

    def test_valid_line_parsed(self):
        """A single valid JSONL line must produce exactly one validated sample."""
        line = self._make_line()
        valid, invalid = parse_generated_lines(line, "director")
        self.assertEqual(len(valid), 1)
        self.assertEqual(invalid, 0)

    def test_instruction_normalised(self):
        """
        The 'instruction' field must be overwritten with the canonical value
        from NODE_CONFIG, regardless of what the model produced.
        """
        line = self._make_line(instruction="some other instruction")
        valid, _ = parse_generated_lines(line, "director")
        self.assertEqual(valid[0]["instruction"], NODE_CONFIG["director"]["instruction"])

    def test_invalid_json_line_skipped(self):
        """Non-JSON lines must increment the invalid counter and be skipped."""
        raw = "not json\n" + self._make_line()
        valid, invalid = parse_generated_lines(raw, "director")
        self.assertEqual(len(valid), 1)
        self.assertEqual(invalid, 1)

    def test_missing_required_keys_skipped(self):
        """A JSON line missing 'output' must be rejected."""
        raw = json.dumps({"instruction": "test", "input": "test"})  # No 'output'
        valid, invalid = parse_generated_lines(raw, "director")
        self.assertEqual(len(valid), 0)
        self.assertEqual(invalid, 1)

    def test_output_as_object_auto_converted(self):
        """
        When the model returns 'output' as a dict instead of a string,
        parse_generated_lines must auto-convert it to a JSON string.
        """
        output_obj = {
            "content": "Plan.",
            "managers": [
                {"name": "M1", "job": "J1"},
                {"name": "M2", "job": "J2"},
            ],
        }
        # Intentionally pass output as dict, not string
        line = json.dumps({"instruction": "test", "input": "test input", "output": output_obj})
        valid, invalid = parse_generated_lines(line, "director")
        self.assertEqual(len(valid), 1)
        self.assertIsInstance(valid[0]["output"], str)

    def test_markdown_fences_stripped(self):
        """
        Lines starting with ``` must be silently skipped — they are
        Markdown code-fence markers, not JSONL data.
        """
        raw = "```json\n" + self._make_line() + "\n```"
        valid, invalid = parse_generated_lines(raw, "director")
        self.assertEqual(len(valid), 1)

    def test_empty_lines_skipped(self):
        """Leading, trailing, and inter-line blank lines must not affect the count."""
        raw = "\n\n" + self._make_line() + "\n\n"
        valid, invalid = parse_generated_lines(raw, "director")
        self.assertEqual(len(valid), 1)
        self.assertEqual(invalid, 0)

    def test_empty_input_skipped(self):
        """A sample with an empty 'input' string must be rejected."""
        line = json.dumps({
            "instruction": "test",
            "input": "",
            "output": json.dumps({
                "content": "Plan.",
                "managers": [{"name": "M1", "job": "J1"}, {"name": "M2", "job": "J2"}],
            }),
        })
        valid, invalid = parse_generated_lines(line, "director")
        self.assertEqual(len(valid), 0)
        self.assertEqual(invalid, 1)

    def test_multiple_valid_lines(self):
        """Three valid JSONL lines must all be parsed successfully."""
        lines = "\n".join([self._make_line(inp=f"project {i}") for i in range(3)])
        valid, invalid = parse_generated_lines(lines, "director")
        self.assertEqual(len(valid), 3)
        self.assertEqual(invalid, 0)


suite = unittest.TestLoader().loadTestsFromTestCase(TestParseGeneratedLines)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

---
## Part 3 — vLLM Samples Tests (`test_vllm_samples.py`)

**Script:** `AI/src/test_vllm_samples.py`

A plain-assertion test suite (no unittest framework) that validates the pure logic of `generate_vllm_samples.py` offline:
- Random prompt generation produces varied output.
- Batch generation prompts are well-structured.
- JSONL parsing correctly accepts/rejects samples.
- Markdown code-fence lines are properly stripped.

Run as a standalone script: `python test_vllm_samples.py`

### 3.1 Setup — Import Tested Functions

In [ ]:
import sys
import os

# Add AI/src to path so we can import generate_vllm_samples directly
AI_SRC_PATH = os.path.abspath("AI/src")
if AI_SRC_PATH not in sys.path:
    sys.path.insert(0, AI_SRC_PATH)

from generate_vllm_samples import (
    get_random_coding_prompt,
    get_random_book_prompt,
    build_generation_prompt,
    parse_generated_lines as parse_generated_lines_vllm,
)

print("generate_vllm_samples functions imported successfully.")

### 3.2 test_random_prompts — Variety and Well-Formedness

Generates 10 coding and 10 book prompts and asserts that they are not all identical and each contains the expected action verbs.

In [ ]:
def test_random_prompts():
    """
    Test that random prompt generation produces varied, well-formed outputs.

    Checks:
      1. 10 coding prompts are not all identical (randomness is working).
      2. 10 book prompts are not all identical.
      3. Each coding prompt contains at least one action verb.
      4. Each book prompt contains at least one narrative verb.
    """
    print("Testing random prompt generation...")

    # Generate sets of prompts
    coding_prompts = [get_random_coding_prompt() for _ in range(10)]
    book_prompts   = [get_random_book_prompt()   for _ in range(10)]

    # Variety check — strict equality would mean randomness is broken
    assert len(set(coding_prompts)) > 1, "Coding prompts should vary"
    assert len(set(book_prompts))   > 1, "Book prompts should vary"

    # Action-verb check for coding prompts
    for prompt in coding_prompts[:3]:
        print(f"  Coding: {prompt}")
        assert any(
            word in prompt.lower()
            for word in ["implement", "create", "build", "develop", "design"]
        ), f"Coding prompt missing action verb: {prompt}"

    # Narrative-verb check for book prompts
    for prompt in book_prompts[:3]:
        print(f"  Book:   {prompt}")
        assert any(
            word in prompt.lower()
            for word in ["write", "create", "develop", "compose", "craft"]
        ), f"Book prompt missing narrative verb: {prompt}"

    print("  ✓ Random prompts are varied and well-formed")


test_random_prompts()

### 3.3 test_prompt_building — Prompt Structure Checks

Verifies that the batch generation prompts contain the required keywords and structural elements, and that the deduplication `avoid_section` is correctly injected when existing inputs are provided.

In [ ]:
def test_prompt_building():
    """
    Test that build_generation_prompt() produces correctly structured prompts.

    Checks:
      1. Coding prompt mentions 'coding' (or a synonym) and 'JSONL', 'prompt', 'response'.
      2. Book prompt mentions 'book' or 'writing' and the same structure fields.
      3. When existing_inputs is provided, the prompt contains a 'NOT reuse' deduplication hint.
    """
    print("\nTesting prompt building...")

    # ── Coding generation prompt ──────────────────────────────────────────────
    coding_prompt = build_generation_prompt("coding", 5, None)
    assert "coding" in coding_prompt.lower(), "Coding prompt should mention 'coding'"
    assert "JSONL"    in coding_prompt, "Coding prompt should mention JSONL format"
    assert "prompt"   in coding_prompt, "Coding prompt should define 'prompt' field"
    assert "response" in coding_prompt, "Coding prompt should define 'response' field"
    print(f"  Coding prompt: {len(coding_prompt)} chars")

    # ── Book generation prompt ────────────────────────────────────────────────
    book_prompt = build_generation_prompt("book", 5, None)
    assert (
        "book" in book_prompt.lower() or "writing" in book_prompt.lower()
    ), "Book prompt should mention 'book' or 'writing'"
    assert "JSONL"    in book_prompt, "Book prompt should mention JSONL format"
    assert "prompt"   in book_prompt, "Book prompt should define 'prompt' field"
    assert "response" in book_prompt, "Book prompt should define 'response' field"
    print(f"  Book prompt:   {len(book_prompt)} chars")

    # ── Deduplication hint ────────────────────────────────────────────────────
    existing = ["existing prompt 1", "existing prompt 2"]
    prompt_with_avoid = build_generation_prompt("coding", 3, existing)
    assert "NOT reuse" in prompt_with_avoid, "Deduplication hint should be present"
    assert "existing prompt 1" in prompt_with_avoid, "Existing inputs should appear in hint"
    print(f"  Prompt with avoidance: {len(prompt_with_avoid)} chars")

    print("  ✓ Prompts are well-structured")


test_prompt_building()

### 3.4 test_sample_parsing — Accept/Reject Logic

Exercises the three key parsing scenarios: all valid, all invalid, and mixed. Verifies exact counts to catch off-by-one errors.

In [ ]:
def test_sample_parsing():
    """
    Test JSONL parsing of generated samples.

    Checks:
      1. Two well-formed samples are both accepted.
      2. Five different failure modes are all rejected (short, missing fields,
         missing prompt, missing response, empty prompt).
      3. A mix of two valid and one invalid is correctly split.
    """
    print("\nTesting sample parsing...")

    # ── Case 1: All valid ─────────────────────────────────────────────────────
    valid_jsonl = (
        '{"prompt": "Task 1", "response": "This is a valid response with enough content to pass"}\n'
        '{"prompt": "Task 2", "response": "Another valid response with sufficient length for validation"}'
    )
    valid, invalid = parse_generated_lines_vllm(valid_jsonl, "coding")
    assert len(valid) == 2, f"Expected 2 valid samples, got {len(valid)}"
    assert invalid == 0,    f"Expected 0 invalid, got {invalid}"
    print(f"  All-valid JSONL:   {len(valid)} accepted, {invalid} rejected")

    # ── Case 2: All invalid ───────────────────────────────────────────────────
    # 5 distinct failure modes:
    #   - response too short (< 50 chars)
    #   - missing 'response' field
    #   - missing 'prompt' field
    #   - not valid JSON
    #   - empty 'prompt' string
    invalid_jsonl = (
        '{"prompt": "Task",   "response": "Short"}\n'
        '{"prompt": "Task 2"}\n'
        '{"response": "Missing prompt"}\n'
        'not even json\n'
        '{"prompt": "", "response": "Empty prompt should be invalid"}'
    )
    valid, invalid = parse_generated_lines_vllm(invalid_jsonl, "coding")
    assert len(valid) == 0, f"Expected 0 valid, got {len(valid)}"
    assert invalid == 5,    f"Expected 5 invalid, got {invalid}"
    print(f"  All-invalid JSONL: {len(valid)} accepted, {invalid} rejected")

    # ── Case 3: Mixed valid and invalid ───────────────────────────────────────
    mixed_jsonl = (
        '{"prompt": "Good task 1", "response": "This is a complete and valid response with enough content"}\n'
        '{"prompt": "Bad",         "response": "Short"}\n'
        '{"prompt": "Good task 2", "response": "Another complete and valid response with sufficient content"}'
    )
    valid, invalid = parse_generated_lines_vllm(mixed_jsonl, "book")
    assert len(valid) == 2, f"Expected 2 valid, got {len(valid)}"
    assert invalid == 1,    f"Expected 1 invalid, got {invalid}"
    print(f"  Mixed JSONL:       {len(valid)} accepted, {invalid} rejected")

    print("  ✓ Parsing correctly validates samples")


test_sample_parsing()

### 3.5 test_markdown_stripping — Code-Fence Handling

LLMs sometimes wrap their JSONL output in Markdown code fences (`` ```json ... ``` ``). This test verifies that the parser strips those lines and still extracts the valid sample.

In [ ]:
def test_markdown_stripping():
    """
    Test that Markdown code-fence lines are stripped before parsing.

    When a model wraps its output in ```json ... ```, the fence lines
    (starting with ```) must be discarded so the embedded JSONL line
    is still parsed correctly.
    """
    print("\nTesting markdown fence handling...")

    markdown_jsonl = (
        "```json\n"
        '{"prompt": "Task", "response": "Valid response with enough content to pass validation"}\n'
        "```"
    )

    valid, invalid = parse_generated_lines_vllm(markdown_jsonl, "coding")
    assert len(valid) == 1, f"Expected 1 valid sample (fences stripped), got {len(valid)}"
    print(f"  Markdown fences stripped: {len(valid)} sample parsed")

    print("  ✓ Markdown handling works correctly")


test_markdown_stripping()

### 3.6 Run All vLLM Tests Together

In [ ]:
def run_all_vllm_tests():
    """
    Run the complete test_vllm_samples.py test suite inline.

    Equivalent to running: python test_vllm_samples.py
    from AI/src/.
    """
    print("=" * 60)
    print("Running generate_vllm_samples.py tests")
    print("=" * 60)

    try:
        test_random_prompts()
        test_prompt_building()
        test_sample_parsing()
        test_markdown_stripping()

        print("\n" + "=" * 60)
        print("✓ All tests passed!")
        print("=" * 60)
    except AssertionError as e:
        print(f"\n✗ Test failed: {e}")
    except Exception as e:
        import traceback
        print(f"\n✗ Unexpected error: {e}")
        traceback.print_exc()


run_all_vllm_tests()

---
## Run All Tests from the Command Line

All tests can also be run without this notebook from `AI/src/`:

```bash
cd AI/src

# Format schema tests (6 test classes, 34 test methods)
python -m unittest tests.test_formats -v

# Sample generation validation tests (26 test methods)
python -m unittest tests.test_generate_samples -v

# vLLM samples functional tests
python test_vllm_samples.py

# Run all unittest tests at once
python -m unittest discover -s tests -v
```

All tests run offline — no Ollama, no vLLM, and no GPU required.